## Import Modules

In [1]:
# PySpark libs
from pyspark.sql import SparkSession, functions as F

# Other libs
from pyspark_helper import row_memory_count

## Set up for Spark

In [2]:
# Set up for Spark app & resource allocation
# (This is a template from DSC 232R final project, actual allocation TBD)
SPARK_CONFIGS = {
    # Resources
    "spark.driver.memory":      "10g",
    "spark.driver.cores" :      "1",
    "spark.executor.instances": "4",
    "spark.executor.cores":     "2",
    "spark.executor.memory":    "38g",

    # Parallelism
    "spark.sql.shuffle.partitions": "32",
    "spark.default.parallelism":    "32",
    "spark.driver.maxResultSize":   "4g"
}

spark = (
    SparkSession.builder
    .appName("steam_reviews_data_subsampling")
    .config(map=SPARK_CONFIGS)
    .getOrCreate()
)

## Define Parameters &

In [6]:
# -----------------------------
# Paths
# -----------------------------
EXPANSE_ROOT = "/expanse/lustre/scratch/bguo3/temp_project"
FULL_PARQUET_PATH = f"file:{EXPANSE_ROOT}/steam_reviews/full_parquet"
SAMPLE_PARQUET_PATH = f"file:{EXPANSE_ROOT}/steam_reviews/subsampled_parquet"

# -----------------------------
# Sampling parameters
# -----------------------------
EVENT_TS_COL = "timestamp_created"
USER_SAMPLE_FRACTION = 0.02  # Subset of 5%
SEED = 42  # Result is reproducible
T_QUANTILE = 0.95  # Use 95th percentile cutoff
RELATIVE_ERROR = 0.001

## Read Full Data from Parquet Files

In [7]:
df_full = spark.read.parquet(FULL_PARQUET_PATH)
print(f"Read full parquet successfully from: {FULL_PARQUET_PATH}")

# Ensure the full data is loaded with all columns intact
df_full.printSchema()

Read full parquet successfully from: file:/expanse/lustre/scratch/bguo3/temp_project/steam_reviews/full_parquet
root
 |-- author_steamid: long (nullable = true)
 |-- appid: integer (nullable = true)
 |-- author_num_games_owned: integer (nullable = true)
 |-- author_num_reviews: integer (nullable = true)
 |-- author_playtime_forever: integer (nullable = true)
 |-- author_playtime_last_two_weeks: integer (nullable = true)
 |-- author_playtime_at_review: integer (nullable = true)
 |-- author_last_played: long (nullable = true)
 |-- review: string (nullable = true)
 |-- voted_up: boolean (nullable = true)
 |-- votes_up: integer (nullable = true)
 |-- votes_funny: long (nullable = true)
 |-- weighted_vote_score: float (nullable = true)
 |-- comment_count: integer (nullable = true)
 |-- written_during_early_access: boolean (nullable = true)
 |-- timestamp_created: long (nullable = true)
 |-- timestamp_updated: long (nullable = true)



## Create User & Time-awared Subsampling

In [5]:
# -----------------------------
# user-aware factor
# -----------------------------
sampled_users = (
    df_full
    .select("author_steamid")
    .where(F.col("author_steamid").isNotNull())
    .distinct()
    .sample(
        withReplacement=False, 
        fraction=USER_SAMPLE_FRACTION, 
        seed=SEED
    )
)

# -----------------------------
# time-aware factor
# -----------------------------
sampled_time_cutoff = df_full.approxQuantile(EVENT_TS_COL, [T_QUANTILE], RELATIVE_ERROR)[0]  # Approximate time that's in T_QUANTILE percentile place
print(f"Chosen cutoff T ({T_QUANTILE:.0%} quantile of {EVENT_TS_COL}): {sampled_time_cutoff}")

# -----------------------------
# Subsampling
# -----------------------------
df_sample = (
    df_full
    .join(F.broadcast(sampled_users), on="author_steamid", how="inner")  # Use broadcast join (for small user table to prevent storage spill OOM issue
    .filter(F.col(EVENT_TS_COL) <= F.lit(sampled_time_cutoff))
)

Chosen cutoff T (95% quantile of timestamp_created): 1690823888.0


## Display General Info

In [8]:
# -----------------------------
# User Info
# -----------------------------
full_user_count = df_full.select("author_steamid").distinct().count()
sample_user_count = sampled_users.count()

print(f"Full distinct users:   {full_user_count}")
print(f"Sample distinct users: {sample_user_count}")

# -----------------------------
# Row & Memory Info
# -----------------------------
for df in [df_full, df_sample]:
    print()
    row_memory_count(df)

Full distinct users:   37918608
Sample distinct users: 758432

Total row counts: 113885601 rows
Total estimated size: 48.57 GB

Total row counts: 2103345 rows
Total estimated size: 0.63 GB


## Write Subsampled Data to Parquet File

In [7]:
(
    df_sample.write
    .mode("overwrite")
    .option("compression", "snappy")
    .parquet(SAMPLE_PARQUET_PATH)
)
print(f"Saved sampled parquet to: {SAMPLE_PARQUET_PATH}")

Saved sampled parquet to: file:/expanse/lustre/scratch/bguo3/temp_project/steam_reviews_parquet/subsampled


In [10]:
spark.stop()